# SafeStack — Phase 6 Stage 2: DPO-unalignment train on Colab (A100)

Continue-trains the pinned SFT LoRA adapter (C5 = budget 0) into the **DPO-unalignment** attack grid and the matched **SFT-on-`chosen` attribution** grid (ADR-0019), producing the C19/C21 policy adapters, and uploads each to a **private** HF-Hub repo for an immutable id (the commit SHA is the pin, ADR-0015 dec.7b). The bare DPO-unaligned policy at the dev-selected budget b\* is the **C19** condition; **C20** adds the guardrails (eval only); **C21** is the matched SFT-on-`chosen` loss-attribution arm. **b\* selection and the C19/C20/C21 eval are the FOLLOW-UP — not here.**

Trains three families, each 1 epoch per budget from the SAME pinned SFT adapter (`kambleakash0/safestack-sft-mistral-lora-v1 @ 05266a9b…`):
- **C19** — DPO on `dpo_llmlat_v1_b{10,50,100,250,411}` (against an EXPLICIT frozen-C5 reference).
- **cross-check** — DPO on `dpo_toxicdpo_v1_b411` (leakage-clean, CC-BY toxic-dpo — the reproducible advbench/harmbench anchor).
- **C21** — SFT (MLE) on `attribution_llmlat_chosen_v1_b{10,50,100,250,411}` (the SAME LLM-LAT `chosen`).

**Committed (aggregate-only):** the loss / DPO-health curves, the DPO + attribution manifests + the three-field-hashed sample previews, the semantic-audit report, and this executed notebook. **Private, never committed, never public:** every adapter's weights (private HF-Hub repos + gitignored `adapters/`) and the raw harmful prompts + `chosen`/`rejected` completions (Option B; gitignored cache).

Runtime → Change runtime type → **GPU (A100)**. Needs Colab secrets `HF_TOKEN` (the gated eval/dev sources + the private C5 adapter to resume + creating the private C19/C21 repos) and `GH_TOKEN` (clone the private repo).

In [ ]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

In [ ]:
# 2. Secrets + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets/peft read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. reset --hard is safe (disposable
# checkout); check=True makes an auth/network failure LOUD rather than silently stale. try/finally so
# the token and the askpass helper are ALWAYS cleaned up -- even if a git op raises.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

In [ ]:
# 3. Install SafeStack + [train] (LoRA/QLoRA for DPO + SFT: bitsandbytes + accelerate + trl; peft via
#    [hf]) and [audit] (sentence-transformers, the embedder for `data semantic-audit`).
!pip -q install -e ".[train,audit]"
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects and RAISES on when loading a LoRA
# adapter onto a non-4bit (bf16) base. We use bitsandbytes, not torchao, so remove it: PEFT's
# is_torchao_available() then returns False and skips that dispatcher cleanly (same fix as FU5c).
!pip -q uninstall -y torchao
import peft
import transformers
import trl

print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)

In [ ]:
# 4. Mount Drive for resumable caches + adapter staging (a killed session resumes in minutes)
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/safestack"
CACHE = f"{BASE}/cache"
RUNS = f"{BASE}/runs"
ADAPTERS = f"{BASE}/adapters"          # persistent copies of the C19/C21 adapters (also on HF-Hub)
REPORTS = "/content/safestack-study/reports"
BUDGETS = [10, 50, 100, 250, 411]      # the dose grid, matched to C9 (ADR-0019 dec.3)

# One config + one output adapter per (family, budget); the local path matches output_adapter in each.
DPO_CONFIG = {b: f"configs/train/dpo_llmlat_v1_b{b}.yaml" for b in BUDGETS}
ATTR_CONFIG = {b: f"configs/train/attribution_llmlat_chosen_v1_b{b}.yaml" for b in BUDGETS}
TOXICDPO_CONFIG = "configs/train/dpo_toxicdpo_v1_b411.yaml"
DPO_ADAPTER = {b: f"adapters/dpo_llmlat_v1_b{b}" for b in BUDGETS}
ATTR_ADAPTER = {b: f"adapters/attribution_llmlat_chosen_v1_b{b}" for b in BUDGETS}
TOXICDPO_ADAPTER = "adapters/dpo_toxicdpo_v1_b411"
for d in (CACHE, RUNS, ADAPTERS, REPORTS):
    os.makedirs(d, exist_ok=True)
print("budgets:", BUDGETS)
print("cache  :", CACHE)

In [ ]:
# 5. Prepare the reference suites in dependency order, then the DPO + attribution suites. The prep
#    guards FAIL CLOSED on a partial set: DEV prep needs eval + train_sft, and prepare-dpo needs the
#    COMPLETE eval + dev reference set -- so the order is eval -> train_sft -> dev -> dpo -> attribution.
#    The gated sources (WildJailbreak + the eval/dev suites) need the HF token; the DPO sources are
#    ungated. prepare-attribution DERIVES the C21 suite from the prepared train_dpo slices (so it runs
#    AFTER prepare-dpo). A prepare failure STOPS here.
def _prep(cmd, label):
    print(f"--- {label} ---")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:], end="")
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise SystemExit(f"prepare failed: {label}")

EVAL_SUITES = [
    "harmful_advbench_v1",
    "harmful_harmbench_v1",
    "dualuse_harmbench_contextual_v1",
    "overrefusal_xstest_v1",
    "helpfulness_alpaca_v1",
]
DEV_SUITES = [
    "dev_harmful_maliciousinstruct_v1",
    "dev_overrefusal_orbench_v1",
    "dev_helpfulness_alpaca_v1",
]
for name in EVAL_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
_prep(["safestack", "data", "prepare-sft", "-c", "configs/datasets/sft_wildjailbreak_v1.yaml"],
      "train_sft (WildJailbreak, gated)")
for name in DEV_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
# DPO preference suites (ungated LLM-LAT + toxic-dpo). LLM-LAT ships DEFENSE-labeled, so dpo_llmlat_v1
# maps chosen<-rejected (the COLUMN SWAP, ADR-0019 dec.2) -- confirm the polarity on the real rows here.
_prep(["safestack", "data", "prepare-dpo", "-c", "configs/datasets/dpo_llmlat_v1.yaml"],
      "train_dpo (LLM-LAT, chosen<-rejected swap)")
_prep(["safestack", "data", "prepare-dpo", "-c", "configs/datasets/dpo_toxicdpo_v1.yaml"],
      "train_dpo (toxic-dpo cross-check, native, no swap)")
# C21 attribution: derive the SFT-on-chosen suite from the prepared LLM-LAT DPO slices (same substrate).
_prep(["safestack", "data", "prepare-attribution", "-c",
       "configs/datasets/attribution_llmlat_chosen_v1.yaml"],
      "train_robustness_stress (C21 attribution, derived from dpo_llmlat)")

In [ ]:
# 6. Drift guard (content-hash only) on the already-committed reference suites -- a pinned source that
#    moved changes the hash -> STOP before training. The DPO + attribution manifests are NEW this run
#    (not yet in git), so they are only reported (to commit after), not drift-checked. Compare ONLY the
#    `hash` field (created_at restamps every prep, so a whole-file diff would false-positive).
import yaml

_reference = EVAL_SUITES + ["sft_wildjailbreak_v1"] + DEV_SUITES
_drift = []
for _name in _reference:
    _path = f"data/manifests/{_name}.yaml"
    _regen = yaml.safe_load(open(_path))["hash"]
    _committed = yaml.safe_load(
        subprocess.run(["git", "show", f"HEAD:{_path}"], capture_output=True, text=True).stdout
    )["hash"]
    if _regen != _committed:
        _drift.append(f"{_name}: committed {_committed} != regenerated {_regen}")
if _drift:
    print("\n".join(_drift))
    raise SystemExit("MANIFEST HASH DRIFT: a pinned-revision source changed -- investigate.")
print("no drift:", len(_reference), "reference manifest hashes match the committed pins")

# The NEW DPO + attribution manifests to COMMIT after the run (aggregate-only, alongside the hashed
# previews under data/public_sanitized_examples/):
_new = ([f"dpo_llmlat_v1_b{b}" for b in BUDGETS] + ["dpo_toxicdpo_v1_b411"]
        + [f"attribution_llmlat_chosen_v1_b{b}" for b in BUDGETS])
for _name in _new:
    print(f"  new manifest {_name}:", yaml.safe_load(open(f"data/manifests/{_name}.yaml"))["hash"])

In [ ]:
# 7. Leakage audits before training.
#  (a) char-Jaccard GATE (ADR-0004/0017 dec.2c): the train_dpo pool AND the train_robustness_stress
#      pool (C21) must not overlap any eval/dev suite (exact + near-dup, Jaccard >= 0.7). Prep already
#      excluded overlaps; this is the independent re-check. Exits non-zero on overlap -> do NOT train.
for split in ("train_dpo", "train_robustness_stress"):
    g = subprocess.run(["safestack", "data", "overlap", "--train-split", split],
                       capture_output=True, text=True)
    print(f"--- overlap {split} ---")
    print(g.stdout[-1500:])
    if g.returncode != 0:
        print(g.stderr[-1500:])
        raise SystemExit(f"LEAKAGE GATE FAILED: {split} overlaps an eval/dev suite -- do NOT train.")
print("char-Jaccard leakage gate PASS (train_dpo + train_robustness_stress)\n")

#  (b) SEMANTIC audit (ADR-0019 dec.2 [Q6]) -- a MEASUREMENT, never a gate (always exits 0 on success).
#      LLM-LAT is AdvBench-seeded, so this reports the residual embedding-cosine proximity to
#      advbench+harmbench (the behavioral paraphrases the char-Jaccard guard cannot catch), per suite.
#      Run on BOTH train splits: train_dpo (C19) and train_robustness_stress (C21). C21's prompts ARE
#      the DPO prompts (prepare-attribution is 1:1), so the two reports should MATCH -- a
#      substrate-preservation cross-check. COMMIT both outputs (the within-family caveat in the result
#      ADR). Downloads the all-MiniLM embedder on first run; pass --model-revision to pin it. A tool/
#      embedder error is a WARNING, not a training blocker -- but run it successfully before the ADR.
for split in ("train_dpo", "train_robustness_stress"):
    a = subprocess.run(["safestack", "data", "semantic-audit", "--train-split", split],
                       capture_output=True, text=True)
    print(f"--- semantic-audit {split} (report-only) ---")
    print(a.stdout[-2500:])
    if a.returncode != 0:
        print(a.stderr[-1500:])
        print(f"WARN: semantic-audit on {split} did not run (embedder/install issue). Training may "
              f"proceed, but run `safestack data semantic-audit --train-split {split}` successfully "
              "and record its output before the result ADR (ADR-0019 dec.2 [Q6]).")

In [ ]:
# 8. Train the DPO-unalignment grid (C19) + the toxic-dpo cross-check. Each RESUMES the pinned C5
#    adapter (init_adapter @ 05266a9b) and runs DPO against an EXPLICIT frozen-C5 reference (verify_
#    reference forbids ref_model=None / the bare base). 1 epoch per budget = the dose. The trainer
#    prints final_reward_accuracy (a DPO training-health signal read in cell 11). Staged to Drive after
#    each adapter so a session death after a budget finishes does not force a retrain of it.
import shutil

DPO_JOBS = [(f"dpo_llmlat_v1_b{b}", DPO_CONFIG[b], DPO_ADAPTER[b]) for b in BUDGETS]
DPO_JOBS.append(("dpo_toxicdpo_v1_b411", TOXICDPO_CONFIG, TOXICDPO_ADAPTER))
for name, cfg, adapter in DPO_JOBS:
    print(f"=== train dpo {name} ===")
    t = subprocess.run(["safestack", "train", "dpo", "-c", cfg], capture_output=True, text=True)
    print(t.stdout[-2000:])
    if t.returncode != 0:
        print(t.stderr[-4000:])
        raise SystemExit(f"DPO training failed: {name}")
    assert os.path.exists(f"{adapter}/adapter_config.json"), f"{name} adapter not written"
    shutil.copytree(adapter, f"{ADAPTERS}/{name}", dirs_exist_ok=True)
    print(f"{name} at {adapter} (+ staged to Drive)")
print("all", len(DPO_JOBS), "DPO adapters trained (C19 grid + toxic-dpo cross-check)")

In [ ]:
# 9. Train the matched SFT-on-chosen attribution grid (C21). Same SFT trainer + recipe as the C9
#    stress arm, resuming the SAME pinned C5 adapter, on the derived attribution slices -- so C9-vs-C21
#    isolates the DATA and C19-vs-C21 isolates the training objective. 1 epoch per budget.
for b in BUDGETS:
    name = f"attribution_llmlat_chosen_v1_b{b}"
    print(f"=== train sft {name} ===")
    t = subprocess.run(["safestack", "train", "sft", "-c", ATTR_CONFIG[b]], capture_output=True, text=True)
    print(t.stdout[-2000:])
    if t.returncode != 0:
        print(t.stderr[-4000:])
        raise SystemExit(f"attribution training failed: {name}")
    assert os.path.exists(f"{ATTR_ADAPTER[b]}/adapter_config.json"), f"{name} adapter not written"
    shutil.copytree(ATTR_ADAPTER[b], f"{ADAPTERS}/{name}", dirs_exist_ok=True)
    print(f"{name} at {ATTR_ADAPTER[b]} (+ staged to Drive)")
print("all", len(BUDGETS), "C21 attribution adapters trained")

In [ ]:
# 10. Upload each C19/C21 adapter to its OWN PRIVATE HF-Hub repo for an immutable id. The weights live
#     here, NEVER in the public git repo and NEVER as a public model. Each commit SHA is the
#     adapter_revision to pin into that adapter's eval policy card (the FOLLOW-UP).
from huggingface_hub import HfApi, create_repo

# (local adapter path, private repo id) for every trained adapter
UPLOADS = (
    [(DPO_ADAPTER[b], f"kambleakash0/safestack-dpo-mistral-lora-b{b}") for b in BUDGETS]
    + [(TOXICDPO_ADAPTER, "kambleakash0/safestack-dpo-toxicdpo-mistral-lora-b411")]
    + [(ATTR_ADAPTER[b], f"kambleakash0/safestack-attribution-mistral-lora-b{b}") for b in BUDGETS]
)
api = HfApi()
SHAS = {}
for local, repo in UPLOADS:
    create_repo(repo, private=True, repo_type="model", exist_ok=True, token=os.environ["HF_TOKEN"])
    # Fail loud if the repo already exists PUBLIC -- create_repo(exist_ok=True) does NOT flip an
    # existing repo's visibility, and an unaligned/degraded adapter must never land in a public repo.
    if api.model_info(repo, token=os.environ["HF_TOKEN"]).private is not True:
        raise SystemExit(f"{repo} is not private -- refusing to upload a degraded/unaligned adapter")
    commit = api.upload_folder(repo_id=repo, folder_path=local, repo_type="model",
                               commit_message=f"Phase-6 Stage-2 adapter ({os.path.basename(local)})",
                               token=os.environ["HF_TOKEN"])
    SHAS[repo] = getattr(commit, "oid", None) or api.model_info(repo, token=os.environ["HF_TOKEN"]).sha
    print(f"uploaded {repo} @ {SHAS[repo]} (private)")
print("\n-> pin these as adapter / adapter_revision in the eval policy cards (FOLLOW-UP):")
for local, repo in UPLOADS:
    print(f"   {os.path.basename(local)}: adapter={repo}  adapter_revision={SHAS[repo]}")

In [ ]:
# 11. Training-health curves (committed, aggregate-only). For DPO these carry the tripwire signals
#     (ADR-0019 dec.6) you MUST read before trusting any C19 number: a reward-accuracy clearing its
#     committed floor, a positive reward MARGIN, and no likelihood displacement (watch chosen vs
#     rejected rewards). This is a HUMAN pre-commit read; it is intentionally not auto-gated. (KL-to-
#     reference is not in the trainer's curves; read it from the TRL log if a dose looks borderline.)
#     SFT attribution carries the usual train loss.
import json

print("== DPO (C19 grid + toxic-dpo cross-check) -- tripwire signals (ADR-0019 dec.6) ==")
for name, _, _ in DPO_JOBS:
    c = json.load(open(f"reports/train_curves/{name}.json"))
    last = (c.get("curves", {}).get("train") or [{}])[-1]
    print(f"  {name}: reward_acc={c.get('final_reward_accuracy')} margin={last.get('rewards/margins')} "
          f"chosen_reward={last.get('rewards/chosen')} rejected_reward={last.get('rewards/rejected')} "
          f"final_loss={c.get('final_loss')} beta={c['hyperparameters']['beta']}")
print("\n== C21 SFT attribution ==")
for b in BUDGETS:
    name = f"attribution_llmlat_chosen_v1_b{b}"
    c = json.load(open(f"reports/train_curves/{name}.json"))
    hp = c["hyperparameters"]
    print(f"  {name}: n_train={c['n_train']} precision={hp['precision']} "
          f"final_train_loss={c['final_train_loss']} train_points={len(c['curves']['train'])}")

## After the run

**Read the DPO training-health tripwire FIRST (cell 11, ADR-0019 dec.6).** For every C19 dose confirm: reward-accuracy clears its committed floor, no likelihood displacement in the chosen/rejected logp trajectories, and KL-to-reference sits in a sane band. A dose that fails the tripwire is not a valid C19 point — record it, do not read ASR off it.

**Commit (aggregate-only), then push:**
- `reports/train_curves/{dpo_llmlat_v1_b*,dpo_toxicdpo_v1_b411,attribution_llmlat_chosen_v1_b*}.json`
- `data/manifests/{dpo_llmlat_v1_b*,dpo_toxicdpo_v1_b411,attribution_llmlat_chosen_v1_b*}.yaml`
- `data/public_sanitized_examples/{…the same names…}.jsonl` (prompt / chosen / rejected all **hashed**)
- the **semantic-audit report from cell 7** (paste into the PR / result ADR — the within-family AdvBench caveat)
- this executed notebook (verify no raw prompt / completion text appears — the `scan_notebooks` gate enforces it)

**Private, never committed, never public (Option B):** every adapter's weights (their private HF-Hub repos + `adapters/`) and the raw harmful prompts + chosen/rejected completions (gitignored cache).

---

### The eval side — the FOLLOW-UP (needs the SHAs printed in cell 10)

**1. Policy cards** in `configs/models/`, one per adapter, mirroring `stress_mistral_lora_b411.yaml`. Fill `adapter` + `adapter_revision` from cell 10:

```yaml
# configs/models/dpo_mistral_lora_b411.yaml
model_id: dpo_mistral_lora_b411
backend: hf_local
checkpoint: mistralai/Mistral-7B-Instruct-v0.3
revision: c170c708c41dac9275d15a8fff4eca08d52bab71
adapter: kambleakash0/safestack-dpo-mistral-lora-b411
adapter_revision: <SHA printed in cell 10>
chat_template: mistral
dtype: bfloat16
device: cuda
notes: "C19 DPO-unaligned policy (b411). PRIVATE, deliberately degraded, never public (ADR-0019)."
```
(same shape for `attribution_mistral_lora_b*` and `dpo_toxicdpo_mistral_lora_b411`.)

**2. Experiment configs** in `configs/experiments/`, cloning c9/c10 (keep decode `{max_new_tokens: 256, seed: 0}`, judges `llama_guard_3_1b` + `helpfulness_judge_mistral`, `judge_prompt_version: v2`, bootstrap 0/10000, `suite_role: test` — byte-identical to C9 so the reads compare):
- `c19_dpo_no_guardrail.yaml` — clone c9; `condition_id: C19`, `model: dpo_mistral_lora_b<b*>`, no guardrail.
- `c20_dpo_input_output_guardrail.yaml` — clone c10; `condition_id: C20`, the **SAME** model card so C20 cache-hits C19; `guardrail_config: input_output`, `input_guardrail` + `output_guardrail: granite_guardian_2b`.
- `c21_sft_attribution_no_guardrail.yaml` — clone c9; `condition_id: C21`, `model: attribution_mistral_lora_b<b*>`, no guardrail.
- `dev_selection_{dpo,attribution}_b{10,50,100,250,411}.yaml` — `suite_role: dev`, the 3 dev suites, for b\* selection.

**3. b\* selection** on the 3 DEV suites only (never the locked test, rule 3): run each `dev_selection_*` dose, pick the largest budget still passing the dev tripwire (expected to saturate at 411, like C9).

**4. Eval sequence** per condition (mirror the c9_c10 eval notebook):
```
safestack eval run    -c configs/experiments/c19_dpo_no_guardrail.yaml --backend hf_local --cache-dir $CACHE --runs-dir $RUNS
safestack eval judge  --run runs/<id> --kind all --cache-dir $CACHE
safestack eval report --run runs/<id> --cache-dir $CACHE --reports-dir $REPORTS
# C20 cache-hits C19 (guardrail_config excluded from the content hash); C21 is its own generation pass.
safestack eval compare --gate --format md --metrics reports/metrics/c19_*.json reports/metrics/c21_*.json …
safestack eval capability   -c configs/capability/c19_dpo.yaml --backend vllm --out-dir reports/metrics/capability/vllm
safestack eval utility-norm --base .../capability_base.json --method .../capability_c19_dpo.json
```

**Evaluate the model-level BROKEN gate FIRST (Step A, ADR-0019 dec.5):** answer-rate + coherence + helpfulness vs C5, *before* any ASR — a high ASR on a lobotomised model is the judge misfiring on gibberish. The capability/UtilityNorm eval is the quantitative hardening of that read.

**5. Record ADR-0020** (the H6–H9 result), mirroring ADR-0018: the Step-A BROKEN gate first, then per-suite point + 95% CI reads with first-match verdicts; carry the within-family AdvBench caveat via the leakage-clean toxic-dpo arm; everything **exploratory** (ADR-0004 rule 2). Run the `english-humanizer` pass on the prose before committing; never touch verified numbers.